# A1 Tensor Product Exploration

Question to keep in mind:

- If I tensor-product two features with the same irreps, does the output stay the same?

Short answer:

- Not automatically.
- A full tensor product expands into every allowed coupling channel.
- You only stay in the same representation type if you explicitly choose `irreps_out` to be that same irrep set.
- Even then, the output values are new bilinear features, not a copy of the input.

This notebook uses the repo's `A1` embedding definitions from `models/local_iso_embedding.py` and the same `FullyConnectedTensorProduct` style used in `models/SR_ocrp.py`.

In [1]:
from collections import Counter
from pathlib import Path
import sys

import torch
from e3nn import o3

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "models").exists():
    repo_root = repo_root.parent
if not (repo_root / "models").exists():
    raise RuntimeError("Could not locate repo root containing a models/ directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from models.local_iso_embedding import (
    build_local_iso_fcc_embedding,
    build_local_iso_hcp_embedding,
)
from models.SR_ocrp import CosineMaskedEquivariantSpatialConv

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. Repo-specific `A1` irreps

In this repo, `irreps_a1` does **not** mean `0e` scalars only.

- `models/local_iso_embedding.py` first computes how many copies of crystal-group `A1` live inside each SO(3) `l` band.
- It then prunes away inactive bands and builds the active feature space `irreps_a1`.

So the model's `A1` feature space is a symmetry-filtered subspace of nonscalar SO(3) irreps.

In [2]:
fcc = build_local_iso_fcc_embedding(device="cpu")
hcp = build_local_iso_hcp_embedding(device="cpu")

print("FCC irreps_a1:", fcc.irreps_a1, "dim=", fcc.irreps_a1.dim)
print("HCP irreps_a1:", hcp.irreps_a1, "dim=", hcp.irreps_a1.dim)
print("FCC irreps_full:", fcc.irreps_full)
print("HCP irreps_full:", hcp.irreps_full)

FCC irreps_a1: 1x4e dim= 9
HCP irreps_a1: 2x2e+1x4e+1x6e dim= 32
FCC irreps_full: 1x2e+1x4e
HCP irreps_full: 2x2e+1x4e+1x6e


In [3]:
def describe_self_product(name, irreps):
    irreps = o3.Irreps(irreps)
    full_tp = o3.FullTensorProduct(irreps, irreps)
    same_tp = o3.FullyConnectedTensorProduct(irreps, irreps, irreps, shared_weights=True)

    print(f"=== {name} ===")
    print("in      :", irreps)
    print("dim     :", irreps.dim)
    print("full out:", full_tp.irreps_out)
    print("full dim:", full_tp.irreps_out.dim)
    print("same out:", same_tp.irreps_out)
    print("weights :", same_tp.weight_numel)
    print()


def count_paths_by_output_irrep(tp):
    counts = Counter()
    for inst in tp.instructions:
        counts[inst.i_out] += 1
    rows = []
    for i_out, mul_ir in enumerate(tp.irreps_out):
        rows.append((i_out, str(mul_ir), counts[i_out]))
    return rows

In [4]:
describe_self_product("scalar 0e", "1x0e")
describe_self_product("pure 4e", "1x4e")
describe_self_product("repo FCC A1", fcc.irreps_a1)
describe_self_product("repo HCP A1", hcp.irreps_a1)

=== scalar 0e ===
in      : 1x0e
dim     : 1
full out: 1x0e
full dim: 1
same out: 1x0e
weights : 1

=== pure 4e ===
in      : 1x4e
dim     : 9
full out: 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
full dim: 81
same out: 1x4e
weights : 1

=== repo FCC A1 ===
in      : 1x4e
dim     : 9
full out: 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
full dim: 81
same out: 1x4e
weights : 1

=== repo HCP A1 ===
in      : 2x2e+1x4e+1x6e
dim     : 32
full out: 4x0e+1x0e+1x0e+4x1e+1x1e+1x1e+4x2e+2x2e+2x2e+1x2e+1x2e+1x2e+1x2e+4x3e+2x3e+2x3e+1x3e+1x3e+1x3e+1x3e+4x4e+2x4e+2x4e+2x4e+1x4e+1x4e+2x4e+1x4e+1x4e+2x5e+2x5e+2x5e+1x5e+1x5e+2x5e+1x5e+1x5e+2x6e+2x6e+2x6e+1x6e+1x6e+2x6e+1x6e+1x6e+2x7e+1x7e+1x7e+2x7e+1x7e+1x7e+2x8e+1x8e+1x8e+2x8e+1x8e+1x8e+1x9e+1x9e+1x9e+1x10e+1x10e+1x10e+1x11e+1x12e
full dim: 1024
same out: 2x2e+1x4e+1x6e
weights : 52



Interpretation:

- `0e x 0e -> 0e`, so the scalar case really does stay scalar even for the full product.
- `4e x 4e` does **not** stay `4e` by default. The full product expands to `0e + 1e + ... + 8e`.
- But `FullyConnectedTensorProduct(4e, 4e, 4e)` is valid because the `4e` channel is one allowed part of that decomposition.
- For repo HCP `A1`, self-product has many available couplings, and projecting back to the same `A1` space still leaves a rich learned bilinear map.

In [5]:
irr = o3.Irreps("1x4e")
full_tp = o3.FullTensorProduct(irr, irr)
same_tp = o3.FullyConnectedTensorProduct(irr, irr, irr, shared_weights=True)

x = torch.randn(3, irr.dim)
out_full = full_tp(x, x)
out_same = same_tp(x, x)

print("x shape       :", tuple(x.shape))
print("full out shape:", tuple(out_full.shape), "irreps_out=", full_tp.irreps_out)
print("same out shape:", tuple(out_same.shape), "irreps_out=", same_tp.irreps_out)
print("mean ||out_same - x||:", (out_same - x).norm(dim=-1).mean().item())
print("max |tp(2x, x) - 2 tp(x, x)|:", (same_tp(2 * x, x) - 2 * out_same).abs().max().item())
print("max |tp(2x, 2x) - 4 tp(x, x)|:", (same_tp(2 * x, 2 * x) - 4 * out_same).abs().max().item())

x shape       : (3, 9)
full out shape: (3, 81) irreps_out= 1x0e+1x1e+1x2e+1x3e+1x4e+1x5e+1x6e+1x7e+1x8e
same out shape: (3, 9) irreps_out= 1x4e
mean ||out_same - x||: 2.4271438121795654
max |tp(2x, x) - 2 tp(x, x)|: 0.0
max |tp(2x, 2x) - 4 tp(x, x)|: 0.0


This cell makes the key distinction explicit:

- `same_tp` returns a feature that **transforms as** `4e`.
- It does **not** return the original `x`.
- It is a learned bilinear coupling of the two inputs.

So “same irreps out” means same representation type, not identity or passthrough.

In [6]:
hcp_same_tp = o3.FullyConnectedTensorProduct(
    hcp.irreps_a1,
    hcp.irreps_a1,
    hcp.irreps_a1,
    shared_weights=True,
)

print("HCP A1 same-out weight_numel:", hcp_same_tp.weight_numel)
print("HCP A1 path counts by output block:")
for row in count_paths_by_output_irrep(hcp_same_tp):
    print(row)

HCP A1 same-out weight_numel: 52
HCP A1 path counts by output block:
(0, '2x2e', 7)
(1, '1x4e', 9)
(2, '1x6e', 8)


In [7]:
fcc_conv = CosineMaskedEquivariantSpatialConv(
    kernel_size=3,
    irreps_in=fcc.irreps_a1,
    irreps_out=fcc.irreps_a1,
    use_residual=False,
)

hcp_conv = CosineMaskedEquivariantSpatialConv(
    kernel_size=3,
    irreps_in=hcp.irreps_a1,
    irreps_out=hcp.irreps_a1,
    use_residual=False,
)

print("FCC conv tp:")
print("  in1:", fcc_conv.tp.irreps_in1)
print("  in2:", fcc_conv.tp.irreps_in2)
print("  out:", fcc_conv.tp.irreps_out)
print("  weight_numel:", fcc_conv.tp.weight_numel)
print()
print("HCP conv tp:")
print("  in1:", hcp_conv.tp.irreps_in1)
print("  in2:", hcp_conv.tp.irreps_in2)
print("  out:", hcp_conv.tp.irreps_out)
print("  weight_numel:", hcp_conv.tp.weight_numel)

FCC conv tp:
  in1: 1x4e
  in2: 1x4e
  out: 1x4e
  weight_numel: 1

HCP conv tp:
  in1: 2x2e+1x4e+1x6e
  in2: 2x2e+1x4e+1x6e
  out: 2x2e+1x4e+1x6e
  weight_numel: 52


## Bottom line

- If you TP two inputs with the same irreps, the output does **not** automatically remain in that same irrep set.
- The unconstrained full product usually expands into many irreps.
- In this repo, layers stay in `irreps_a1` because we explicitly construct `FullyConnectedTensorProduct(irreps_a1, irreps_a1, irreps_a1)`.
- That preserves the **feature type** seen by later equivariant layers, but it still computes a new learned bilinear feature.
- For FCC `A1 = 1x4e`, that same-out map is especially narrow: one coupling path.
- For HCP `A1 = 2x2e+1x4e+1x6e`, same-out TP is richer: many coupling paths back into the same `A1` space.

Useful next experiment:

- Compare `FullyConnectedTensorProduct(irreps_a1, irreps_a1, irreps_a1)` against a wider `irreps_out` and see which added channels help downstream decoding.